In [15]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.compose import make_column_transformer
from ipywidgets import interact
import ipywidgets as widgets
## these are all the imports that we used

In [16]:
# reading file
traindf = pd.read_csv('https://raw.githubusercontent.com/wang-anna6/databootcamp/refs/heads/main/train.csv')
pd.set_option('display.max_columns', None) ## We did this so that we could see all of the columns when looking through the data
traindf.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1024,120,RL,43.0,3182,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Blmngtn,Norm,Norm,TwnhsE,1Story,7,5,2005,2006,Gable,CompShg,VinylSd,VinylSd,BrkFace,14.0,Gd,TA,PConc,Gd,Gd,No,GLQ,16,Unf,0,1330,1346,GasA,Ex,Y,SBrkr,1504,0,0,1504,0,0,2,0,2,1,Gd,7,Typ,1,Gd,Attchd,2005.0,Fin,2,437,TA,TA,Y,156,20,0,0,0,0,NaN,NaN,NaN,0,5,2008,WD,Normal,191000
1,811,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,6,6,1974,1999,Hip,CompShg,HdBoard,HdBoard,BrkFace,99.0,TA,TA,CBlock,TA,TA,No,ALQ,663,LwQ,377,0,1040,GasA,Fa,Y,SBrkr,1309,0,0,1309,1,0,1,1,3,1,Gd,5,Typ,1,Fa,Attchd,1974.0,RFn,2,484,TA,TA,Y,265,0,0,0,0,648,Fa,GdPrv,NaN,0,1,2006,WD,Normal,181000
2,1385,50,RL,60.0,9060,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Norm,Norm,1Fam,1.5Fin,6,5,1939,1950,Gable,CompShg,WdShing,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,TA,Mn,Rec,204,Unf,0,356,560,GasA,TA,Y,SBrkr,698,560,0,1258,0,0,1,0,2,1,TA,6,Typ,0,NaN,Detchd,1939.0,Unf,1,280,TA,TA,P,0,0,0,0,0,0,NaN,MnPrv,NaN,0,10,2009,WD,Normal,105000
3,627,20,RL,NaN,12342,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,5,1960,1978,Hip,CompShg,Wd Sdng,Wd Sdng,NaN,0.0,TA,TA,CBlock,TA,TA,No,Unf,0,Unf,0,978,978,GasA,TA,Y,SBrkr,1422,0,0,1422,0,0,1,0,3,1,TA,6,Min1,1,TA,Attchd,1960.0,RFn,1,286,TA,TA,Y,0,0,36,0,0,0,NaN,GdWo,Shed,600,8,2007,WD,Normal,139900
4,814,20,RL,75.0,9750,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Gable,CompShg,MetalSd,MetalSd,BrkFace,243.0,TA,TA,CBlock,TA,TA,No,Rec,608,Unf,0,834,1442,GasA,Gd,Y,SBrkr,1442,0,0,1442,0,0,1,1,4,1,TA,7,Typ,0,NaN,Attchd,1958.0,RFn,1,301,TA,TA,Y,0,0,275,0,0,0,NaN,NaN,Shed,500,4,2007,COD,Normal,157900


In [ ]:
missing = traindf[traindf.columns[traindf.isna().sum() > 0]]
nums_missing = missing.select_dtypes(['float', 'int']).columns.tolist()
cat_missing = missing.select_dtypes('object').columns.tolist()
traindf['HouseAge'] = traindf['YrSold'] - traindf['YearBuilt'] ## here I wanted to create a sort of age for the house. I wanted to find the amount of years it took for the house to be sold after it was built because I thought it would be a good predictor of price.

In [ ]:
imputer = make_column_transformer((SimpleImputer(), nums_missing),
                                  (SimpleImputer(strategy = "most_frequent"), cat_missing),
                                  remainder = 'passthrough',
                                  verbose_feature_names_out=False)
imputer.fit_transform(traindf)
## in order to deal with missing values, we used a simple imputer to replace the missing values of catagorical columns with the most common catagory in that list. For quantitative columns the simple imputer fills the missing values with the mean

array([[43.0, 14.0, 2005.0, ..., 'Normal', 191000, 3],
       [78.0, 99.0, 1974.0, ..., 'Normal', 181000, 32],
       [60.0, 0.0, 1939.0, ..., 'Normal', 105000, 70],
       ...,
       [60.0, 0.0, 1957.0, ..., 'Normal', 115000, 51],
       [55.0, 0.0, 1925.0, ..., 'Normal', 189950, 89],
       [53.0, 130.0, 2007.0, ..., 'Normal', 174000, 2]],
      shape=(1095, 82), dtype=object)

In [ ]:
X = traindf[['LotArea', 'YrSold', 'OverallQual', 'OverallCond', 'SaleCondition', 'PoolArea', 'Neighborhood', 'Utilities','GrLivArea','BsmtCond','1stFlrSF','HouseAge','MSSubClass']]
y = traindf['SalePrice']
## each of the features here are ones that we thought would make a good predictor of price in a house.

In [ ]:

cat_feats = X.select_dtypes(include='object').columns.tolist()
num_feats = X.select_dtypes(exclude='object').columns.tolist()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=22)
## here we are splitting the training data so we have some data to test the model with

In [ ]:
ohe = OneHotEncoder(drop='first')
transform = make_column_transformer((ohe, cat_feats), remainder = PolynomialFeatures(include_bias=False))
## we one hot encoded the data to transform all the catagorical features into numeric values in order to be able to work with them easier

In [ ]:
pipe = Pipeline([('transform', transform), ("scale", StandardScaler()), ('model', LinearRegression())])

## Here is our pipeline that we made to organize our processes and prevent us from writing the same code by hand multiple times

In [13]:
# fitting data to pipeline
pipe.fit(X_train,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('transform',
                 ColumnTransformer(remainder=PolynomialFeatures(include_bias=False),
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='first'),
                                                  ['SaleCondition',
                                                   'Neighborhood', 'Utilities',
                                                   'BsmtCond'])])),
                ('scale', StandardScaler()), ('model', LinearRegression())])

In [14]:
test_rmse = np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))
train_rmse = np.sqrt(mean_squared_error(y_train, pipe.predict(X_train)))
print("test rmse is",test_rmse.round(), "train rmse", train_rmse.round())
## this was the lowest RMSE we could get

test rmse is 26544.0 train rmse 24868.0
